# Fully-connected Neural Networks - Human Activity Recognition
Neil John Catapang

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

from time import time

In [ ]:
# Load the subject data
subject_train = pd.read_csv('UCI HAR Dataset/train/subject_train.txt', header=None, names=['Subject'])
subject_test = pd.read_csv('UCI HAR Dataset/test/subject_test.txt', header=None, names=['Subject'])

subject_train.head()

In [ ]:
# Load the train and test data
X_train = pd.read_csv('UCI HAR Dataset/train/X_train.txt', header=None, 
                      sep=r'\s+')
X_test = pd.read_csv('UCI HAR Dataset/test/X_test.txt', header=None, 
                      sep=r'\s+')
y_train = pd.read_csv('UCI HAR Dataset/train/y_train.txt', header=None, 
                      sep=r'\s+', names=['Activity'])
y_test = pd.read_csv('UCI HAR Dataset/test/y_test.txt', header=None, 
                      sep=r'\s+', names=['Activity'])

# Get feature names
columns = pd.read_csv('UCI HAR Dataset/features.txt', header=None, 
                      sep=r'\s+')
columns_list = columns[1].to_list()
X_train.columns = columns_list
X_test.columns = columns_list

display(y_train[0:5])
X_train.head()


## Exploratory Data Analysis

### Check for Columns with Null Counts

In [ ]:
nulls = X_train.isnull().sum()
print(nulls[nulls > 0])
print(X_train.isnull().sum().sum(), '\n')
X_train.info()

In [ ]:
nulls = X_test.isnull().sum()
print(nulls[nulls > 0])
print(X_test.isnull().sum().sum(), '\n')
X_test.info()

No columns with null counts are detected.

### Value Counts of Subject and Activity

In [ ]:
plt.figure(figsize=(15, 8))

# Train Data - Activity
plt.subplot(221)
sns.countplot(data=y_train, x='Activity')
plt.title("Activity - Train")

# Train Data - Subjects
plt.subplot(222)
sns.countplot(data=subject_train, x='Subject')
plt.title("Subject - Train")

# Test Data - Activity
plt.subplot(223)
sns.countplot(data=y_test, x='Activity')
plt.title("Activity - Test")

# Test Data - Subjects
plt.subplot(224)
sns.countplot(data=subject_test, x='Subject')
plt.title("Subject - Test")

plt.subplots_adjust(hspace=0.3)
plt.show()

The classes are quite well-balanced to some degree. The train and test subjects are also balanced. Class balancing may not be required for this dataset.

### Correlation Heatmap for Means of Features

In [ ]:
# Get the values of means of features
feature_means = [s for s in columns_list if 'mean' in s.lower()]
corr_subset = X_train[feature_means].corr()
print(f'Number of Features with Mean: {len(feature_means)}')

# Heatmap
plt.figure(figsize=(30, 30))
sns.heatmap(corr_subset, annot=False, cmap='coolwarm')
plt.title("Correlation Heatmap - Feature Means")
plt.show()

We can see that some of the features are highly correlated with each other. However, we want to keep all the features in this dataset since fully-connected neural networks are non-linear models which can handle highly-correlated features well. I will also attempt to apply dimensionality reduction using autoencoders, so removing features here may be unnecessary.

## Fully-connected Neural Network

In [ ]:
y_train = y_train.to_numpy().flatten()
y_test = y_test.to_numpy().flatten()

In [ ]:
# Scale the data before training model (no balancing)
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(max_iter=200, random_state=42, 
                          solver='adam', early_stopping=True,
                          n_iter_no_change=10, learning_rate_init=0.0001,
                          validation_fraction=0.08))
])

param_grid = {
    'mlp__hidden_layer_sizes': [(256, 128, 32)],
    'mlp__activation': ['tanh', 'relu'],
    'mlp__alpha': [0.0001, 0.001, 0.01]
}

kfoldcv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
grid1 = GridSearchCV(pipeline, param_grid, scoring='accuracy', cv=kfoldcv, verbose=3, n_jobs=-1)

start1 = time() 
grid1.fit(X_train, y_train)
print("GridSearchCV took %.2f seconds." % (time() - start1))

In [ ]:
best_model_1 = grid1.best_estimator_
print("Neural Network")
print("Best Parameters:", grid1.best_params_)
print("Best Accuracy:", grid1.best_score_)

In [ ]:
# ANN-GridSearch - Accuracy and Confusion Matrix
print('Neural Network')
print(f"Accuracy = {best_model_1.score(X_test, y_test)}")
print('Confusion Matrix')
y_test_pred1 = best_model_1.predict(X_test)
print(confusion_matrix(y_test, y_test_pred1))

# ANN-GridSearch - Classification Report
print(classification_report(y_test, y_test_pred1))

## Fully-connected Neural Network and Balanced Dataset

In [ ]:
# Balance the dataset
oversample = SMOTE(random_state=42)
X_train_over, y_train_over = oversample.fit_resample(X_train, y_train)

In [ ]:
grid2 = GridSearchCV(pipeline, param_grid, scoring='accuracy', cv=kfoldcv, verbose=3, n_jobs=-1)

start2 = time() 
grid2.fit(X_train_over, y_train_over)
print("GridSearchCV took %.2f seconds." % (time() - start2))

In [ ]:
best_model_2 = grid2.best_estimator_
print("Neural Network + SMOTE")
print("Best Parameters:", grid2.best_params_)
print("Best Accuracy:", grid2.best_score_)

In [ ]:
# ANN-SMOTE - Accuracy and Confusion Matrix
print('Neural Network + SMOTE')
print(f"Accuracy = {best_model_2.score(X_test, y_test)}")
print('Confusion Matrix')
y_test_pred2 = best_model_2.predict(X_test)
print(confusion_matrix(y_test, y_test_pred2))

# ANN-SMOTE - Classification Report
print(classification_report(y_test, y_test_pred2))

## Fully-connected Neural Network + Balanced Dataset + Autoencoder

### Autoencoder

In [ ]:
# Use keras for autoencoder implementation
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import MinMaxScaler

# Scale the data
scaler = MinMaxScaler()   # Used since the output of autoencoder is from 0 to 1 (sigmoid activation)
X_train_scaled = scaler.fit_transform(X_train_over)
X_test_scaled = scaler.transform(X_test)

# Create the layers of autoencoder
input_layer = Input(shape=(X_train_scaled.shape[1],))   # Input layer
encoder = Dense(256, activation='relu')(input_layer) 
bottleneck = Dense(64, activation='linear')(encoder)  # Bottleneck - Actual dimensions after dim. red.
decoder = Dense(256, activation='relu')(bottleneck)
output_layer = Dense(X_train_scaled.shape[1], activation='sigmoid')(decoder)

# Create the autoencoder
autoencoder_full = Model(inputs=input_layer, outputs=output_layer)
autoencoder_dimred = Model(inputs=input_layer, outputs=bottleneck)

# Add early stopping to avoid overfit
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

# Compile and train the autoencoder
autoencoder_full.compile(optimizer=Adam(), loss='mse')
autoencoder_full.fit(X_train_scaled, X_train_scaled,
                epochs=1000,
                batch_size=128,
                shuffle=True,
                validation_split=0.15,
                callbacks=[early_stopping])

In [ ]:
# Apply dimensionality reduction using the encoder
X_train_reduced = autoencoder_dimred.predict(X_train_scaled)
X_test_reduced = autoencoder_dimred.predict(X_test_scaled)

print(X_train_reduced.shape)
print(X_test_reduced.shape)

### Neural Network

In [ ]:
grid3 = GridSearchCV(pipeline, param_grid, scoring='accuracy', cv=kfoldcv, verbose=3, n_jobs=-1)

start3 = time() 
grid3.fit(X_train_reduced, y_train_over)
print("GridSearchCV took %.2f seconds." % (time() - start3))

In [ ]:
best_model_3 = grid3.best_estimator_
print("Neural Network + SMOTE + Autoencoder")
print("Best Parameters:", grid3.best_params_)
print("Best Accuracy:", grid3.best_score_)

In [ ]:
# ANN-SMOTE-AE - Accuracy and Confusion Matrix
print('Neural Network + SMOTE + Autoencoder')
print(f"Accuracy = {best_model_3.score(X_test_reduced, y_test)}")
print('Confusion Matrix')
y_test_pred3 = best_model_3.predict(X_test_reduced)
print(confusion_matrix(y_test, y_test_pred3))

# ANN-SMOTE-AE - Classification Report
print(classification_report(y_test, y_test_pred3))